In [18]:
import pandas as pd

df_train = pd.read_csv('train_d.csv')
df_test = pd.read_csv('test_d.csv')

amino_acids = [
    "alanine", "arginine", "asparagine", "aspartic acid", "cysteine",
    "glutamic acid", "glutamine", "glycine", "histidine", "isoleucine",
    "leucine", "lysine", "methionine", "phenylalanine", "proline",
    "serine", "threonine", "tryptophan", "tyrosine", "valine"
]

filtered_df = df_train[df_train['names'].apply(lambda x: any(amino in x for amino in amino_acids))]
filtered_df2 = df_test[df_test['names'].apply(lambda x: any(amino in x for amino in amino_acids))]

new_df = pd.concat([filtered_df, filtered_df2], ignore_index=True)

new_df.to_csv('filtered_aminoacids.csv', index=False)


missing_amino_acids = [aa for aa in amino_acids if not new_df['names'].str.contains(aa).any()]
print(missing_amino_acids)

['histidine']


<h4> Exercise5*: Consider the following data

- Choose at least five physicochemical properties of amino acids.
- Represent each sequence as a vector based on the selected properties of its amino acids.
- Standardize the data before training.
- Reduce the number of features using F-score and Incremental Feature Selection (IFS)
- Build an SVM model and remember to tune its hyperparameters.
- Evaluate your model on the test data using different performance metrics - for K, P, R, T separately. 
- Compare your results with the literature. Make commments.

In [ ]:
import numpy as np
from sklearn.feature_selection import f_classif
import pandas as pd

#Data
X = np.array([
    [2.5, 0.3, 1.1, 4.2],
    [2.4, 0.4, 1.0, 4.1],
    [2.3, 0.2, 1.3, 4.0],
    [2.2, 0.5, 1.2, 4.3],
    [2.6, 0.3, 1.1, 4.1],
    [3.5, 1.3, 3.1, 2.2],
    [3.4, 1.4, 3.0, 2.1],
    [3.6, 1.2, 3.3, 2.0],
    [3.3, 1.5, 3.2, 2.3],
    [3.7, 1.3, 3.1, 2.4],
])

#labels
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

#F-score and p-value
F, p = f_classif(X, y)

#Print the results
features = ['F1', 'F2', 'F3', 'F4']
df = pd.DataFrame({
    'Feature': features,
    'F-score': F,
    'p-value': p
})

print(df.sort_values(by='F-score', ascending=False))

In [2]:
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction, molecular_weight

seq = Seq("ATGCGGCTAGC")

In [4]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
my_seq = (
    "MAEGEITTFTALTEKFNLPPGNYKKPKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQ"
    "LSAESVGEVYIKSTETGQYLAMDTSGLLYGSQTPSEECLFLERLEENHYNTYTSKKHAKN"
    "WFVGLKKNGSCKRGPRTHYGQKAILFLPLPV"
)
analysed_seq = ProteinAnalysis(my_seq)
print(analysed_seq.molecular_weight())
print(analysed_seq.gravy())
print(analysed_seq.count_amino_acids())


16974.047700000003
-0.5781456953642389
{'A': 6, 'C': 3, 'D': 5, 'E': 11, 'F': 6, 'G': 14, 'H': 5, 'I': 5, 'K': 12, 'L': 18, 'M': 2, 'N': 7, 'P': 8, 'Q': 6, 'R': 6, 'S': 10, 'T': 13, 'V': 5, 'W': 1, 'Y': 8}


In [55]:
#sekwencje
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Przykładowa sekwencja białkowa (możesz ją zmienić na własną)
seqs = ["ILRRTASAPAKGRKKSKMGFQEMVEIK",
            "KDIEGKENSLAEDKDGRRKGKASIKDP",
            "WFVGLKKNGSCKRGPRTHYGQKAILFLPLPV"] 

# Analiza
vecs = []
for s in seqs:
    s = ProteinAnalysis(s)
    vecs.append([s.molecular_weight(), s.isoelectric_point(), s.instability_index(), s.gravy(), s.aromaticity(), s.secondary_structure_fraction()[0], s.secondary_structure_fraction()[1], s.secondary_structure_fraction()[2]]) #, ])
    #masa cząsteczkowa, wartość izoelektryczna, wskaźnik niestabilności, wskaźnik hydrofobowości (gravy), zawartość reszt aromatycznych, procent helisy alfa, procent struktury beta, procent zwoju (coil)
# print(vecs)

print(vecs[0])
print(vecs[1])
print(vecs[2])
# print("Długość sekwencji:", len(sequence))
# print("Aminokwasy (zliczenie):", analyzed_seq.count_amino_acids())
# print("Częstość względna (frakcje):", analyzed_seq.get_amino_acids_percent())
# print("Średnia masa reszty:", analyzed_seq.molecular_weight() / len(sequence))
# print("Wzór sumaryczny:", analyzed_seq.get_amino_acids_percent())


[3061.6707, 11.122295188903806, 79.42962962962962, -0.7111111111111111, 0.037037037037037035, 0.4814814814814815, 0.18518518518518517, 0.2222222222222222]
[2985.2670000000003, 8.336358451843264, 38.52962962962963, -1.7925925925925925, 0.0, 0.4444444444444444, 0.4074074074074074, 0.1111111111111111]
[3512.180500000001, 10.57740840911865, 51.5, -0.2225806451612902, 0.12903225806451613, 0.29032258064516125, 0.29032258064516125, 0.3870967741935484]


Amino acids features

In [ ]:
#download data index
import requests

url = "https://www.genome.jp/ftp/db/community/aaindex/aaindex1"
response = requests.get(url)

with open("aaindex1", "wb") as f:
    f.write(response.content)

#parse_aaindex1_modified was copied from chatGPT; its possible to use biopython-1.78 to unpack the data but it's painful
def parse_aaindex1_modified(path):
    aaindex = {}
    aa_order = []  # final order of single-letter amino acids
    current_entry = {}
    current_key = None
    reading_values = False
    value_lines = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("H "):
                current_key = line[2:].strip()
                current_entry = {"description": "", "values": {}}
            elif line.startswith("D "):
                current_entry["description"] = line[2:].strip()
            elif line.startswith("I "):
                # Extract amino acid pairs and flatten them into a list of letters
                aa_pairs = line[1:].strip().split()
                aa_order = []
                for pair in aa_pairs:
                    aa_order.extend(pair.split("/"))
            elif line == "//":
                if current_key and value_lines:
                    # Flatten the value lines
                    values = []
                    for vl in value_lines:
                        try:
                            values += [float(x) for x in vl.strip().split()]
                        except ValueError:
                            continue
                    if len(values) == 20:
                        current_entry["values"] = dict(zip(aa_order, values))
                        aaindex[current_key] = current_entry
                # Reset state for next entry
                current_entry = {}
                value_lines = []
                aa_order = []
                reading_values = False
                current_key = None
            elif line and (line[0].isdigit() or line[0] == '-' or line[0] == ' '):
                value_lines.append(line)
    
    return aaindex

aaindex = parse_aaindex1_modified("aaindex1")
#check if the aaindex is loaded correctly
print(f"Records found: {len(aaindex)}\n")

for key, record in aaindex.items():
    if "Hydrophobicity" in record["description"]:
        print("ID:", key)
        print("Desctiption:", record["description"])
        print("Hydrophobicity Ala:", record["values"].get("A"))
        break



Records found: 552

ID: ARGP820101
Desctiption: Hydrophobicity index (Argos et al., 1982)
Hydrophobicity Ala: 0.61


In [ ]:
import numpy as np

def sequence_to_feature_matrix(sequence, aaindex_data):
    # dict: AAindex ID -> understandable name
    selected_features = 
    {
    "ARGP820101": "hydrophobicity",
    "CHAM820101": "side chain volume",
    "FAUJ880111": "molecular weight",
    "JANJ780101": "isoelectric point",
    "KYTJ820101": "helical propensity",
    "ROSM880102": "solvent accessible surface area",
    "ZIMJ680104": "molecular volume",
    "GRAR740102": "electric charge",
    "BHAR880101": "polarity",
    "OOBM770101": "side chain mobility",
    "TANS770106": "side chain spatial volume",
}

    feature_matrix = []

    for aa in sequence.upper():
        features = []
        for idx in selected_features:
            value = aaindex_data.get(idx, {}).get("values", {}).get(aa, None)
            features.append(value if value is not None else np.nan)
        feature_matrix.append(features)

    return feature_matrix
print(sequence_to_feature_matrix("ACDEFGHIKLMNPQRSTVWY", aaindex))

[[0.61, 0.046, 0.0, 27.8, 1.8, -0.67, 6.0, 8.1, 0.357, -1.895, 0.937], [0.61, 0.23, 1.0, 50.7, -3.2, 1.09, 7.59, 10.4, 0.323, -1.755, 1.085], [0.47, 0.151, 0.0, 68.2, -3.5, 1.78, 3.22, 12.3, 0.497, -1.535, 0.679], [1.18, 0.221, 0.0, 33.5, 1.9, -1.67, 5.74, 5.7, 0.295, -1.963, 0.886], [0.07, 0.0, 0.0, 24.5, -0.4, 0.0, 5.97, 9.0, 0.544, -1.898, 0.901], [1.95, 0.131, 0.0, 51.5, -1.6, -1.75, 6.3, 8.0, 0.509, -1.699, 0.748], [0.05, 0.108, 0.0, 45.0, -0.7, -0.42, 5.66, 8.6, 0.444, -1.767, 1.487], [1.88, 0.298, 0.0, 55.2, -1.3, 0.98, 5.66, 6.2, 0.42, -1.686, 1.227], [0.46, 0.105, 0.0, 60.6, -3.5, 1.57, 2.77, 13.0, 0.511, -1.518, 1.64], [0.6, 0.291, 1.0, 94.7, -4.5, 3.89, 10.76, 10.5, 0.529, -1.475, 1.725], [0.0, 0.18, 0.0, 68.7, -3.5, 2.12, 5.65, 10.5, 0.493, -1.521, 1.078], [1.07, 0.128, 0.0, 15.5, 2.5, -2.0, 5.05, 5.5, 0.346, -2.035, 1.004], [2.22, 0.186, 0.0, 22.8, 4.5, -3.02, 6.02, 5.2, 0.462, -1.951, 0.178], [1.53, 0.186, 0.0, 27.6, 3.8, -3.02, 5.98, 4.9, 0.365, -1.966, 0.808], [0.06, 0.

In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Sekwencja białkowa do analizy
sequence = "MVKVYAPASSANMSVGFDVLGAAVTPVDGALLGDVVTVEAAETFSLNNLGQKL"
analyzed_seq = ProteinAnalysis(sequence)

# Punkt izoelektryczny całej sekwencji
sequence_pI = analyzed_seq.isoelectric_point()

# Definicje grup aminokwasów
groups = {
    'alkalinity': {'R', 'H', 'K'},
    'acidity': {'D', 'E'},
    'pH_neutral': {'N', 'Q', 'S', 'T', 'Y', 'C'},
    'polarity': {'A', 'V', 'L', 'I', 'M', 'F', 'W', 'P', 'G'},
}

# Skala hydrofobowości Kyte-Doolittle
hydropathy_index = {
    'A': 1.8,  'C': 2.5,  'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5,  'K': -3.9, 'L': 3.8,
    'M': 1.9,  'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2,  'W': -0.9, 'Y': -1.3,
}

# Przygotowanie listy cech dla każdego aminokwasu w sekwencji
aa_features_list = []

for aa in sequence:
    features = [1 if aa in groups['alkalinity'] else 0, 
                1 if aa in groups['acidity'] else 0,
                1 if aa in groups['pH_neutral'] else 0,
                0 if aa in groups['polarity'] else 1,
                hydropathy_index.get(aa, 0.0)
                ]

    aa_features_list.append(features)

print(aa_features_list)


[{'aminokwas': 'M', 'zasadowość': 0, 'kwasowość': 0, 'polarność obojętna': 0, 'niepolarny': 1, 'hydropathy': 1.9, 'pI': 4.104764366149902}, {'aminokwas': 'V', 'zasadowość': 0, 'kwasowość': 0, 'polarność obojętna': 0, 'niepolarny': 1, 'hydropathy': 4.2, 'pI': 4.104764366149902}, {'aminokwas': 'K', 'zasadowość': 1, 'kwasowość': 0, 'polarność obojętna': 0, 'niepolarny': 0, 'hydropathy': -3.9, 'pI': 4.104764366149902}, {'aminokwas': 'V', 'zasadowość': 0, 'kwasowość': 0, 'polarność obojętna': 0, 'niepolarny': 1, 'hydropathy': 4.2, 'pI': 4.104764366149902}, {'aminokwas': 'Y', 'zasadowość': 0, 'kwasowość': 0, 'polarność obojętna': 1, 'niepolarny': 0, 'hydropathy': -1.3, 'pI': 4.104764366149902}, {'aminokwas': 'A', 'zasadowość': 0, 'kwasowość': 0, 'polarność obojętna': 0, 'niepolarny': 1, 'hydropathy': 1.8, 'pI': 4.104764366149902}, {'aminokwas': 'P', 'zasadowość': 0, 'kwasowość': 0, 'polarność obojętna': 0, 'niepolarny': 1, 'hydropathy': -1.6, 'pI': 4.104764366149902}, {'aminokwas': 'A', 'zas

In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Przykładowa sekwencja białkowa (możesz zmienić)
sequence = "MVKVYAPASSANMSVGFDVLGAAVTPVDGALLGDVVTVEAAETFSLNNLGQKL"

# Zainicjalizuj analizator
analyzed_seq = ProteinAnalysis(sequence)
aa_counts = analyzed_seq.count_amino_acids()
aa_total = sum(aa_counts.values())

# Definicje grup aminokwasów
group_definitions = {
    "Polarne zasadowe (R, H, K)": ['R', 'H', 'K'],
    "Polarne kwasowe (D, E)": ['D', 'E'],
    "Polarne obojętne (N, Q, S, T, Y, C)": ['N', 'Q', 'S', 'T', 'Y', 'C'],
    "Niepolarne (A, V, L, I, M, F, W, P, G)": ['A', 'V', 'L', 'I', 'M', 'F', 'W', 'P', 'G'],
}

# Oblicz liczebność i procent dla każdej grupy
print("Grupowanie aminokwasów według właściwości chemicznych:\n")
for group_name, aa_list in group_definitions.items():
    group_count = sum(aa_counts.get(aa, 0) for aa in aa_list)
    group_percent = group_count / aa_total * 100
    print(f"{group_name}: {group_count} reszt ({group_percent:.2f}%)")
